# 09. BBC Few Shot Research Extension

**Evidence driven lab.** Summarize 25 recorded model fits across 5 per category sample sizes and 5 random seeds. These measurements originate from the authentic historical BBC training data experiment. This notebook does not rerun expensive training or invent a new experiment.

The student must run this notebook from the `ITAI2373-NewsBot-Final` project root or directly from `notebooks/`. The original copyrighted BBC article bodies are not required to inspect the reported results.


In [ ]:
from pathlib import Path
import csv
import statistics
root = Path.cwd()
if not (root / 'data/results/fewshot_learning_curve.csv').exists():
    root = root.parent
path = root / 'data/results/fewshot_learning_curve.csv'
assert path.exists(), 'Open the notebook inside the NewsBot project folder'
with path.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 25, 'Expected five sample sizes and five seeds per size'
print('Loaded', len(rows), 'recorded historical BBC learning-curve experiments')


## Evaluation methodology

The original study used stratified training/test partitioning after text deduplication. For each reduced training size (5, 10, 25, 50 or 100 labeled examples **per BBC category**) it drew five seed-controlled subsets from the **same training partition**. Each fitted a TF IDF and Logistic Regression pipeline, then evaluated accuracy and macro F1 on a fixed 288 article holdout.

**Important:** This is low-data *supervised* news classification. It is not few-shot prompting of a generative transformer, not novel category discovery, and not proof of news generalization. Repeated access to the same holdout is exploratory.


In [ ]:
sizes = [5,10,25,50,100]
summary = []
for size in sizes:
    records = [r for r in rows if int(r['per_class_samples']) == size]
    seeds = {int(r['seed']) for r in records}
    assert len(records) == 5 and len(seeds) == 5
    f1 = [float(r['holdout_macro_f1']) for r in records]
    acc = [float(r['holdout_accuracy']) for r in records]
    summary.append({'articles_per_category': size,
                    'total_labeled_training': 5 * size,
                    'mean_macro_f1': statistics.mean(f1),
                    'sd_macro_f1': statistics.stdev(f1),
                    'mean_accuracy': statistics.mean(acc)})
for r in summary:
    print(f"{r['articles_per_category']:3d}/category | macro F1 {r['mean_macro_f1']*100:5.2f}% ± {r['sd_macro_f1']*100:4.2f}% | accuracy {r['mean_accuracy']*100:5.2f}%")


## Learning curve visualization

A plot is optional and uses Matplotlib if installed. The live NewsBot web app also renders the corresponding data directly without Matplotlib or a new server dependency.


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Matplotlib is optional; numerical evidence printed above remains available.')
else:
    xs = [v['total_labeled_training'] for v in summary]
    ys = [v['mean_macro_f1'] * 100 for v in summary]
    spread = [v['sd_macro_f1'] * 100 for v in summary]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.errorbar(xs, ys, yerr=spread, marker='o', capsize=4)
    ax.set_xlabel('Total BBC training articles, five categories')
    ax.set_ylabel('Heldout macro F1 (%)')
    ax.set_title('Measured BBC reduced training sample learning curve')
    ax.grid(True, alpha=.2)
    plt.show()


## Reproduce the underlying experiment

Reproducing the model fits requires your authorized local BBC training and test CSVs. The source is `scripts/evaluate_bbc.py`, specifically `run_experiment(..., enable_fewshot=True)`. Example terminal command:

```bash
python scripts/evaluate_bbc.py --train '/local/BBC News Train.csv' --test '/local/BBC News Test.csv' --out data/results/
```

Keep the original BBC article bodies out of the public GitHub repo. Changing the dataset, model, preprocessing, or seed can alter the recorded metrics. The 25 committed rows are evidence of the existing recorded experiment, **not** newly generated scores from this notebook.
